In [1]:
from core.config import config
from rich import print as rprint




python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8


In [2]:
def get_llm():
    """返回 LLM 实例，未配置 key 时返回 None。"""
    if not config.LLM_API_KEY:
        return None
    from langchain.chat_models import init_chat_model
    return init_chat_model(
        model=config.LLM_MODEL,
        api_key=config.LLM_API_KEY,
        base_url=config.LLM_BASE_URL,
        temperature=0,
        model_provider=config.MODEL_PROVIDER,
        extra_body={"thinking": {"type": "disabled"}}
    )


llm = get_llm()
response = llm.invoke("翻译如下的汉字：你好世界")
rprint(response)

AIMessage(
    content='你好世界（Nǐ hǎo shìjiè）翻译成英文是 **"Hello, 
World"**。\n\n这是编程领域中最经典的示例短语，通常用于测试程序是否能正常运行。😊',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 46,
            'prompt_tokens': 256,
            'total_tokens': 302,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}
        },
        'model_provider': 'openai',
        'model_name': 'mimo-v2.5',
        'system_fingerprint': None,
        'id': '90f0a011e9bd41519b0f879bb7439328',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f335a-8e6d-71a1-a48b-89c5b83ee920-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 256,
        'output_tokens': 46,
        'total_tokens': 302,
        'input_token_details': {'cache_read': 192},
        'output_token_details': {'reasoning': 0}
    }
)

In [3]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
in_memory_saver = InMemorySaver()


cur_thread_id = "1"

agent = create_agent(
    model=llm,
    checkpointer=in_memory_saver
)
config = {"configurable": {"thread_id": cur_thread_id}}

response1 = agent.invoke(
    {"messages": [HumanMessage("你好，我是小明")]},
    config=config
)

for msg in response1["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你好，我是小明
================================== Ai Message ==================================

你好，小明！很高兴认识你！😊

我是MiMo，由小米大模型Core团队开发的AI助手。有什么我可以帮助你的吗？无论是回答问题、聊天，还是其他需要，我都很乐意为你服务！


In [4]:
response2 = agent.invoke(
    {"messages": [HumanMessage("你好，我叫什么？")]},
    config=config
)
for msg in response2["messages"]:
    msg.pretty_print()


================================ Human Message =================================

你好，我是小明
================================== Ai Message ==================================

你好，小明！很高兴认识你！😊

我是MiMo，由小米大模型Core团队开发的AI助手。有什么我可以帮助你的吗？无论是回答问题、聊天，还是其他需要，我都很乐意为你服务！
================================ Human Message =================================

你好，我叫什么？
================================== Ai Message ==================================

你好！你叫**小明**呀！😊

刚才你已经告诉我了，我记得哦～有什么我可以帮你的吗？


In [5]:

from agents.schemas import IntentDecision
from langchain.agents.structured_output import ToolStrategy
in_memory_saver = InMemorySaver()
ROUTER_PROMPT = """
你是电商导购助手的主路由。根据【最近对话历史 + 业务记忆 + 本次问题】把请求分类为唯一 task_type：
- shopping: 商品推荐/对比/查找/SKU 与商品事实问题。
- knowledge: 与具体商品有关的知识问答（用法、成分、科普）。
- chitchat: 闲聊、问候、无关问题。
- unknown: 无法判断。

请你给出：task_type、confidence（0-1，越高越自信）、reason（简短理由）。


""".strip()



router = create_agent(
    model=llm,
    checkpointer=in_memory_saver,
    system_prompt=ROUTER_PROMPT,
    response_format=ToolStrategy(IntentDecision),
)

response = router.invoke(
    {"messages": [HumanMessage("你好啊")]},
    config={"configurable": {"thread_id": cur_thread_id}}
)

rprint(response)



{
    'messages': [
        HumanMessage(
            content='你好啊',
            additional_kwargs={},
            response_metadata={},
            id='f5b57857-f389-4aa5-a3e3-d97f595b4817'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 470,
                    'total_tokens': 543,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 448}
                },
                'model_provider': 'openai',
                'model_name': 'mimo-v2.5',
                'system_fingerprint': None,
                'id': '4ffb878c75bd4d3b98bbb1792ac3f3f1',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f335a-bef2-76a3-b720-9dab611af577-0',
            tool_calls=[
                {
                    'name': 'IntentDecision',
                    'args': {
                        'task_type': 'chitchat',
                        'confidence': 0.95,
                        'reason': 
'用户发送的是简单的问候语"你好啊"，属于闲聊场景，没有涉及任何商品推荐、知识问答或购物相关的内容。'
                    },
                    'id': 'call_e88496b2b1c046529b6645d1',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 470,
                'output_tokens': 73,
                'total_tokens': 543,
                'input_token_details': {'cache_read': 448},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Returning structured response: task_type=\'chitchat\' confidence=0.95 
reason=\'用户发送的是简单的问候语"你好啊"，属于闲聊场景，没有涉及任何商品推荐、知识问答或购物相关的内容。\'',
            name='IntentDecision',
            id='cad9d1a7-fbb8-48f8-98e2-f2391602c74b',
            tool_call_id='call_e88496b2b1c046529b6645d1'
        )
    ],
    'structured_response': IntentDecision(
        task_type='chitchat',
        confidence=0.95,
        reason='用户发送的是简单的问候语"你好啊"，属于闲聊场景，没有涉及任何商品推荐、知识问答或购物相关的内容。'
    )
}

In [6]:
from rag.retriever import Retriever
from rag.models import RetrievalPlan, RetrievalOutput
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
checkpoint = InMemorySaver()
SEARCH_PROMPT = """
你是一个电商导购助手。
你需要根据用户的导购意图，从商品库中搜索候选商品为用户进行推荐。
要求：
1. 只返回真实存在的商品，不要编造商品。
2. 如果没有找到合适商品，返回空列表。
""".strip()

python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:129: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(uri=self.milvus_url)
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:139: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(name=self.collection_name, schema=schema, consistency_level="Strong")
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:140: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instea

In [7]:

@tool
def search_product(question: str) -> RetrievalOutput:
    """
    根据用户问题搜索去向量库召回商品，返回商品列表。
    """
    plan = RetrievalPlan(query=question, top_k=4, search_mode="hybrid")
    retriever = Retriever()
    
    retrieval_output = retriever.retrieve(plan)
    
    
    return retrieval_output
 
 
 
 
 
 
agent = create_agent(
            model = llm,
            checkpointer=checkpoint,
            system_prompt=SEARCH_PROMPT,
            tools= [search_product]
        )

response = agent.invoke(
    {"messages": [HumanMessage("推荐一款200元以内适合油皮的防晒")]},
    config={"configurable": {"thread_id": cur_thread_id}}
)
rprint(response)

G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:129: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(uri=self.milvus_url)
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:139: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(name=self.collection_name, schema=schema, consistency_level="Strong")
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:140: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index("sparse_vector", {"index_type": "SPARSE_INVERTED_INDEX", "metric_type": "IP"})
G:\dev\welove-shop-agt\backend\ai-service\rag\vector_store.py:141: PyMilvusDeprecationWarning: `Collection.create_index` is an O

{
    'messages': [
        HumanMessage(
            content='推荐一款200元以内适合油皮的防晒',
            additional_kwargs={},
            response_metadata={},
            id='99256561-ffff-4b5e-82c9-e5e84b7a8ccf'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 30,
                    'prompt_tokens': 319,
                    'total_tokens': 349,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256}
                },
                'model_provider': 'openai',
                'model_name': 'mimo-v2.5',
                'system_fingerprint': None,
                'id': 'e645454177d2413b8bb97df8870114c2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f335a-d548-7a12-835a-9467e116e084-0',
            tool_calls=[
                {
                    'name': 'search_product',
                    'args': {'question': '200元以内适合油皮的防晒'},
                    'id': 'call_1ebb078cb8e24dec9f38a022',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 319,
                'output_tokens': 30,
                'total_tokens': 349,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content="plan=RetrievalPlan(intent='knowledge_qa', query='200元以内适合油皮的防晒', 
search_targets=['knowledge'], category=None, category_id=None, product_id=None, brand=None, budget_min=None, 
budget_max=None, skin_type=None, season=None, positive_requirements=[], negative_requirements=[], doc_types=[], 
chunk_types=['text'], top_k=4, search_mode='hybrid') results=[SearchResult(content='5. 中性皮肤\\n- 
特征：水油平衡，毛孔细腻，状态理想\\n- 护理重点：维持现状，做好基础保湿和防晒\\n\\n## 
护肤步骤（早晚通用基础版）\\n1. 洁面 → 2. 爽肤水 → 3. 精华 → 4. 眼霜 → 5. 乳液/面霜 → 6. 防晒（白天）', 
metadata=ChunkMetadata(doc_id=4, chunk_id=None, product_id=None, category_id=0, source='护肤知识-肤质分类与护理', 
title='护肤知识-肤质分类与护理', doc_type='md', chunk_type='text', page=None, chunk_index=1, total_chunks=0, 
content_hash='', extra={}), score=0.016393441706895828, dense_score=0.016393441706895828, sparse_score=None, 
rerank_score=None), SearchResult(content='# 场景化护肤建议\\n\\n## 去海南/海边旅行护肤\\n- 
**防晒是重中之重**：SPF50+ PA++++的防水防晒霜，每2小时补涂\\n- 
**晒后修复**：携带芦荟胶或含积雪草的修复面膜，晒后立即使用\\n- 
**精简护肤**：旅行期间减少功效型产品，以保湿修复为主\\n- 
**必备清单**：高倍防晒、芦荟胶、保湿喷雾、卸妆油、洁面乳\\n- 
**避免**：旅行期间不要尝试新产品，不要做果酸焕肤\\n\\n## 夏季护肤\\n- 换用清爽型水乳，减少面霜用量\\n- 
加强防晒，选择轻薄不闷痘的防晒产品\\n- 控油产品适当使用，但不要过度清洁\\n- 补水面膜每周2-3次\\n\\n## 冬季护肤\\n- 
增加保湿力度，可用精华油叠加面霜\\n- 洁面换成温和的氨基酸洁面\\n- 身体乳不能省，沐浴后立即涂抹\\n- 
唇膏和护手霜随身携带\\n\\n## 熬夜急救\\n- 熬夜前：涂一层厚厚的睡眠面膜\\n- 熬夜后：洁面 → 补水面膜 → 维C精华提亮 → 
保湿面霜\\n- 内调：多喝水，补充维生素B族\\n\\n## 换季敏感\\n- 精简护肤步骤，暂停功效型产品\\n- 
使用修复类精华（含神经酰胺、积雪草）\\n- 避免频繁更换产品\\n- 必要时就医，不要自行用药', 
metadata=ChunkMetadata(doc_id=5, chunk_id=None, product_id=None, category_id=0, source='护肤知识-场景护肤建议', 
title='护肤知识-场景护肤建议', doc_type='md', chunk_type='text', page=None, chunk_index=0, total_chunks=0, 
content_hash='', extra={}), score=0.016129031777381897, dense_score=0.016129031777381897, sparse_score=None, 
rerank_score=None), SearchResult(content='# 肤质分类与护理指南\\n\\n## 五大肤质类型\\n\\n### 1. 干性皮肤\\n- 
特征：毛孔细小，皮肤紧绷，容易脱皮、产生细纹\\n- 护理重点：深层保湿、滋润修复\\n- 
推荐成分：透明质酸、神经酰胺、角鲨烷、甘油\\n- 避免：含酒精的控油产品、皂基洁面\\n\\n### 2. 油性皮肤\\n- 
特征：毛孔粗大，T区出油明显，容易长痘、黑头\\n- 护理重点：控油保湿、清洁毛孔\\n- 
推荐成分：水杨酸、烟酰胺、茶树精油、高岭土\\n- 避免：过于滋润的面霜、厚重的防晒\\n\\n### 3. 混合性皮肤\\n- 
特征：T区偏油，两颊偏干，最常见的肤质\\n- 护理重点：分区护理，T区控油，两颊保湿\\n- 推荐产品：清爽型水乳 

In [8]:
from assistant.graph import AssistantGraph

ModuleNotFoundError: No module named 'langchain.checkpoint'

In [ ]:


llm = get_llm()
question = "油皮的防晒通常有成分？"
graph = AssistantGraph(llm)
conversation_id = "1"
# 直接await，不要套asyncio.run()
result = await graph.run(
    question=question,
    conversation_id=conversation_id,
)
print(result)